# 5?. XGBoost ??

*?? ??? ?? ? ???? ??? ??? ??(nbviewer.org)? ??? ?? ??(colab.research.google.com)?? ??? ? ????.*

<table class="tfo-notebook-buttons" align="left">
  <td>
    <a target="_blank" href="https://nbviewer.org/github/rickiepark/handson-gb/blob/main/Chapter05/Advanced_XGBoost_Unveiled.ipynb"><img src="https://jupyter.org/assets/share.png" width="60" />??? ??? ??? ??</a>
  </td>
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/rickiepark/handson-gb/blob/main/Chapter05/Advanced_XGBoost_Unveiled.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />?? ??(Colab)?? ????</a>
  </td>
</table>

In [ ]:
# ???? ???? ?? ??? ?????.
import sys
if 'google.colab' in sys.modules:
    !pip install -q --upgrade xgboost
    !wget -q https://raw.githubusercontent.com/rickiepark/handson-gb/main/Chapter05/atlas-higgs-challenge-2014-v2.csv.gz

In [ ]:
# ?? ??
import warnings
warnings.filterwarnings('ignore')

import xgboost as xgb
xgb.set_config(verbosity=0)

## XGBoost ?? ???

### ?? ????

In [ ]:
import pandas as pd
import numpy as np
from sklearn import datasets
iris = datasets.load_iris()

In [ ]:
df = pd.DataFrame(data=np.c_[iris['data'], iris['target']],
                  columns=iris['feature_names'] + ['target'])

In [ ]:
df.head()

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
# ???? ?? ??? ??? ??? ????.
X_train, X_test, y_train, y_test = train_test_split(iris['data'], 
                                                    iris['target'], random_state=2)

#### XGBoost ?? ??

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

In [ ]:
xgb = XGBClassifier(booster='gbtree', objective='multi:softprob', 
                    max_depth=6, learning_rate=0.1, n_estimators=100, 
                    n_jobs=-1)
xgb.fit(X_train, y_train)
y_pred = xgb.predict(X_test)
score = accuracy_score(y_pred, y_test)
print('??: ' + str(score))

`accuracy_score()` ?? ?? `score()` ???? ??? ? ????.

In [ ]:
xgb.score(X_test, y_test)

XGBoost? ?? ??? API? ???? ?? ???(Booster) ??? `predict()` ???? `multi:softprob`? ? ??? ???? `multi:softmax`? ? ??? ???? ?????.

In [ ]:
import xgboost as xgb

dtrain = xgb.DMatrix(X_train, y_train)
dtest = xgb.DMatrix(X_test[:5])

param = {'objective': 'multi:softprob', 'num_class': 3}
bstr = xgb.train(param, dtrain, 10)
bstr.predict(dtest)

In [ ]:
param = {'objective': 'multi:softmax', 'num_class': 3}
bstr = xgb.train(param, dtrain, 10)
bstr.predict(dtest)

### ??? ????

In [ ]:
X, y = datasets.load_diabetes(return_X_y=True)

#### XGBoost ?? ?? (?? ??)

In [ ]:
from sklearn.model_selection import cross_val_score
from xgboost import XGBRegressor

xgb = XGBRegressor(booster='gbtree', objective='reg:squarederror', 
                   max_depth=6, learning_rate=0.1, n_estimators=100, 
                   n_jobs=-1)

scores = cross_val_score(xgb, X, y, 
                         scoring='neg_mean_squared_error', cv=5)

# ?? ??? ???? ??????.
rmse = np.sqrt(-scores)

# RMSE? ?????.
print('RMSE:', np.round(rmse, 3))

# ?? ??? ?????.
print('RMSE ??: %0.3f' % (rmse.mean()))

In [ ]:
pd.DataFrame(y).describe()

## ?? ?? ?? - ????

### ???

In [ ]:
df = pd.read_csv('atlas-higgs-challenge-2014-v2.csv.gz', nrows=250000, compression='gzip')
df.head()

In [ ]:
del df['Weight']
del df['KaggleSet']
df = df.rename(columns={"KaggleWeight": "Weight"})
label_col = df['Label']
del df['Label']
df['Label'] = label_col

##### ??? ???? ?? ???? ??? ??

In [ ]:
df_h = pd.read_csv('atlas-higgs-challenge-2014-v2.csv.gz', nrows=250000, compression='gzip')

In [ ]:
df_new = df_h.drop(['Weight', 'KaggleSet', 'Label'], axis=1).assign(Label=df_h['Label'])
df_new = df_new.rename(columns={'KaggleWeight': 'Weight'})

In [ ]:
df_new.head()

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df['Label'].replace(('s', 'b'), (1, 0), inplace=True)

In [ ]:
X = df.iloc[:,1:31]
y = df.iloc[:,-1]

##### ?? ???? ?? ??

In [ ]:
X_new = df_new.loc[:, ~df_new.columns.isin(['EventId','Weight','Label'])]
y_new = df_new.loc[:, 'Label']

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_new = le.fit_transform(df_new['Label'])
y_new

In [ ]:
print(le.classes_)

### ?? ??

In [ ]:
xgb = XGBClassifier(n_estimators=5)
xgb.fit(X_train, y_train, 
        eval_set=[(X_train, y_train), (X_test, y_test)],
        eval_metric='mlogloss')

In [ ]:
xgb.evals_result()

### ???

In [ ]:
df['test_Weight'] = df['Weight'] * 550000 / len(y)

In [ ]:
s = np.sum(df[df['Label']==1]['test_Weight'])
b = np.sum(df[df['Label']==0]['test_Weight'])

In [ ]:
b/s

### ??

In [ ]:
import xgboost as xgb

# ??? ??? xgboost.DMatrix ???. -999.0? ??? ??? ????.
xgmat = xgb.DMatrix(X, y, missing = -999.0, weight=df['test_Weight'])

# xgboost ???? ??
param = {}
# ??? ???? ??? ???? ??? ???? ?? ??? ?????.
param['objective'] = 'binary:logitraw'
# ?? ??? ???? ?????.
param['scale_pos_weight'] = b/s
param['eta'] = 0.1
param['max_depth'] = 6
param['eval_metric'] = 'auc'

# ams ??? ?????.
plst = list(param.items())+[('eval_metric', 'ams@0.15')]

watchlist = [(xgmat,'train')]

# 120? ?? ???
num_round = 120

print('??? ?? ??, ?? ??? ??')
bst = xgb.train(plst, xgmat, num_round, watchlist)
bst.save_model('higgs.model')
print('?? ??')

##### ???? API? ????

In [ ]:
clf = XGBClassifier(n_estimators=120, learning_rate=0.1, missing=-999.0, 
                    scale_pos_weight=b/s)

clf.fit(X, y, sample_weight=df['test_Weight'], 
        eval_set=[(X, y)], eval_metric=['auc', 'ams@0.15'],
        sample_weight_eval_set=[df['test_Weight']])

clf.save_model('higgs-sklearn.model')

In [ ]:
clf.evals_result()

In [ ]:
clf.score(X, y)